In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 25
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
16.190981711884675

Trial 1 =========================================
15.908849831676967

Trial 2 =========================================
13.489125363231851

Trial 3 =========================================
13.731540734479582

Trial 4 =========================================
15.268190121671568

Trial 5 =========================================
13.455705326022716

Trial 6 =========================================
13.127518989671259

Trial 7 =========================================
14.18118446263351

Trial 8 =========================================
13.362596527694048

Trial 9 =========================================
15.78762184308299

Trial 10 =========================================
13.551817130875005

Trial 11 =========================================
15.475278473289247



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 12 =========================================
15.533942467350526

Trial 13 =========================================
16.704322108808714

Trial 14 =========================================
15.876375562083751

Trial 15 =========================================
13.503767268079983

Trial 16 =========================================
14.614297571253788

Trial 17 =========================================
13.653226629508623

Trial 18 =========================================
14.492708442730095

Trial 19 =========================================
17.221953069182614

Trial 20 =========================================
15.080405899584576

Trial 21 =========================================
16.412580267782193

Trial 22 =========================================
13.798715099964426

Trial 23 =========================================
13.342549057063627

Trial 24 =========================================
13.480128536971312

Trial 25 =========================================
16.047940129328616

Trial 

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


Trial 82 =========================================
15.510300391879435

Trial 83 =========================================
13.564423775574461

Trial 84 =========================================
14.901660798606182

Trial 85 =========================================
15.060513158262033

Trial 86 =========================================
13.60497697080301

Trial 87 =========================================
13.8404170252767

Trial 88 =========================================
15.007582138926134

Trial 89 =========================================
14.0603685241921

Trial 90 =========================================
15.26927863100183

Trial 91 =========================================
15.374592576461126

Trial 92 =========================================
13.680518672148322

Trial 93 =========================================
15.533942467350526

Trial 94 =========================================
14.803879739238326

Trial 95 =========================================
14.883072514383066

Trial 96 ===

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 97 =========================================
15.855457569040093

Trial 98 =========================================
15.94523920712006

Trial 99 =========================================
14.026100370085942

[16.19098171 15.90884983 13.48912536 13.73154073 15.26819012 13.45570533
 13.12751899 14.18118446 13.36259653 15.78762184 13.55181713 15.47527847
 15.53394247 16.70432211 15.87637556 13.50376727 14.61429757 13.65322663
 14.49270844 17.22195307 15.0804059  16.41258027 13.7987151  13.34254906
 13.48012854 16.04794013 15.86457822 14.37176744 14.02859474 13.7990953
 13.01972799 13.72919533 14.99955535 14.27431842 15.74886655 15.36230493
 13.7372691  13.48112619 14.76588413 15.99273784 13.23935976 15.53394247
 15.15493208 13.1613235  13.81492956 14.98961757 13.7835937  15.53394247
 16.80415418 13.95040533 15.65605808 13.97148349 16.1891142  13.4884314
 13.31268632 13.5759443  13.13351022 14.74464691 15.62899913 13.36897382
 15.53889723 17.62372375 13.64872822 14.19799521 14.5141514 

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 17.62372374677195
Avg = 14.717862628458471
Std = 1.0932821358028768


In [6]:
print(y_max_arr.tolist())

[16.190981711884675, 15.908849831676967, 13.489125363231851, 13.731540734479582, 15.268190121671568, 13.455705326022716, 13.127518989671259, 14.18118446263351, 13.362596527694048, 15.78762184308299, 13.551817130875005, 15.475278473289247, 15.533942467350526, 16.704322108808714, 15.876375562083751, 13.503767268079983, 14.614297571253788, 13.653226629508623, 14.492708442730095, 17.221953069182614, 15.080405899584576, 16.412580267782193, 13.798715099964426, 13.342549057063627, 13.480128536971312, 16.047940129328616, 15.864578223886465, 14.371767438664406, 14.02859473763198, 13.799095297044795, 13.019727987394022, 13.729195334273289, 14.999555345748503, 14.274318424608733, 15.748866552805007, 15.362304932932203, 13.73726910213286, 13.481126186977912, 14.765884132989301, 15.992737839510575, 13.239359764145343, 15.533942467350526, 15.154932078364185, 13.161323496429208, 13.81492955531295, 14.989617568520332, 13.783593699302322, 15.533942467350526, 16.804154182538507, 13.950405330738358, 15.6

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-25.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)